# PySpark 7: aggregations and windows

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-07-aggregations-and-windows.ipynb)

Finance wants revenue by region with subtotals, product wants each customer's latest order and their sessions on
the site, and the CRM team wants loyalty streaks. Some of these collapse rows into groups; others need every row
kept but compared with its neighbours.

The first half of this lesson aggregates: `groupBy` with many functions at once, conditional and approximate
aggregates, `pivot`, `rollup`, `cube` and grouping sets. The second half uses window functions: totals per group
on every row, rankings and top-N, `lag` and `lead`, running and moving frames, the latest record per key,
sessions, and gaps-and-islands.

Each section explains how something works, where it matters and what to watch out for, then runs small examples. Read each output, then the **What to notice** note under it. Run the cells in order.

**Prerequisites:** [PySpark 6: joins](pyspark-06-joins.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `customers` (one row per customer; customer 4 has no region, and 7 and 8 have never ordered), `orders` (one row per order; order 12 lands just after midnight UTC), `order_items` (one row per order line; each order's lines add up to its amount), `products` (one row per SKU; `SKU-4014` is missing, so one order line has no product) and `events` (one row per user event; one event is an exact retry): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#setup-and-sample-data).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F
from pyspark.sql import Window

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),  # a bulk reorder
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing: an order line with no product row
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)),
    (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)),
    (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)),
    (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)),
    (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)),
    (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),
    (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

In [ ]:
import contextlib
import io


def plan_of(df, mode="formatted"):
    """Return the text `df.explain(mode)` prints, so a cell can pick out the lines that matter."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue()

## Group and aggregate

`df.groupBy(cols).agg(...)` collapses rows into one row per group. Pass several aggregate functions to one `agg`
so the data is scanned and shuffled once. Common functions: `count("*")` (rows), `count(col)` (non-null values),
`countDistinct`, `sum`, `avg`, `min`, `max`, `collect_list` (all values, as an array) and `collect_set` (distinct
values).

Spark aggregates in two steps: each task computes a **partial** result for its rows, the partial results are
shuffled by the group key, and a **final** step combines them. Only one partial row per key and task crosses the
network.

**Where it matters**
- **Daily and per-customer metrics** in gold tables.
- **Data profiling:** row counts, distinct counts and ranges per column or group.

**Watch out for**
- `avg` ignores nulls; if missing means zero, fill first.
- Groups with no rows do not appear. Start from the dimension with a `left` join when every customer must be listed.
- `collect_list` on a large group builds one huge array in one task.

### Example

Summarize orders per customer with five aggregates in one pass, and look at the two-step plan.

In [ ]:
per_customer = orders.groupBy("customer_id").agg(
    F.count("*").alias("orders"),
    F.sum("amount").alias("revenue"),
    F.round(F.avg("amount"), 2).alias("avg_order"),
    F.max("order_ts").alias("last_order_at"),
    F.collect_set("status").alias("statuses"),
)
per_customer.orderBy("customer_id").show(truncate=False)
per_customer.explain()

**What to notice:** One row per customer who has orders: customers 7 and 8 are absent. The plan shows the two steps: an aggregate
with `partial_count`, `partial_sum` and the other partial functions before the `Exchange`, and the final
aggregate after it. The operator is `ObjectHashAggregate` rather than `HashAggregate` because `collect_set` keeps
a set object per group. `collect_set` returned each customer's distinct statuses in no guaranteed order.

## Conditional aggregates

To compute several filtered metrics in one pass, put the condition inside the aggregate:
`F.sum(F.when(cond, col))` or, in SQL, `sum(amount) FILTER (WHERE status = 'paid')`. `F.count_if(cond)` counts rows
where a condition is true. This avoids running one filtered query per metric and joining the results.

**Where it matters**
- **KPI tables:** revenue, returns and open orders side by side per day.
- **Rates:** returned orders divided by all orders, per customer.

**Watch out for**
- `F.when(cond, col)` without `otherwise` gives null for other rows, which `sum` ignores (good) but `count(col)` also skips (check that is what you want).

### Example

Compute paid revenue, returned revenue, the number of open orders and a return rate per customer in one aggregation.

In [ ]:
orders.groupBy("customer_id").agg(
    F.sum(F.when(F.col("status") == "paid", F.col("amount"))).alias("paid_revenue"),
    F.expr("sum(amount) FILTER (WHERE status = 'returned')").alias("returned_revenue"),
    F.count_if(F.col("status") == "placed").alias("open_orders"),
    F.round(F.count_if(F.col("status") == "returned") / F.count("*"), 2).alias("return_rate"),
).orderBy("customer_id").show()

**What to notice:** Customer 1 has 120.00 of paid revenue, 30.00 returned and a return rate of 0.33 (one of three orders). Customers
with no returns show null returned revenue, because a sum of no values is null, not zero; wrap it in `coalesce`
if the report should show 0.

## Approximate aggregates

Exact distinct counts and percentiles need every value in one place, which is expensive on large data.
`approx_count_distinct(col, rsd)` uses a HyperLogLog sketch with a relative error of about 5 percent by default,
and `percentile_approx(col, p)` returns a value close to the true percentile. `median` and `percentile` are exact.

**Where it matters**
- **Dashboards over billions of events:** distinct users per day, p95 latency.
- **Profiling** large tables before choosing partition keys.

**Watch out for**
- Approximate results are not additive: you cannot sum daily approximate distinct counts into a monthly one.
- Use exact functions for finance and billing.

### Example

Compare exact and approximate distinct counts and percentiles on 100,000 generated rows.

In [ ]:
clicks = spark.range(100_000).select((F.col("id") % 7919).alias("user_id"), (F.col("id") % 1000).alias("latency_ms"))
clicks.agg(
    F.countDistinct("user_id").alias("exact_users"),
    F.approx_count_distinct("user_id").alias("approx_users"),
    F.approx_count_distinct("user_id", rsd=0.01).alias("approx_users_rsd_1pct"),
    F.median("latency_ms").alias("exact_median"),
    F.percentile_approx("latency_ms", 0.5).alias("approx_median"),
    F.percentile_approx("latency_ms", 0.95).alias("approx_p95"),
).show()

**What to notice:** The default approximate distinct count is several percent away from the exact 7,919: its `rsd` of 0.05 is a
typical relative error, not a guaranteed bound. Asking for `rsd=0.01` lands much closer, at the cost of a larger
sketch. The approximate median is an actual value from the data next to the exact 499.5. On billions of rows the
approximate versions use a small, fixed amount of memory per group, which is why they exist.

## Pivot and unpivot

`groupBy(row_key).pivot(column, values).agg(...)` turns the values of one column into columns: one row per key,
one column per value. Always pass the list of values; otherwise Spark runs an extra job to find them, and a new
value appearing in the data silently adds a column. `unpivot` (alias `melt`) turns columns back into rows.

**Where it matters**
- **Wide reports** for spreadsheets and BI tools: one column per status or month.
- **Normalizing wide vendor files** with one column per month into a long table.

**Watch out for**
- Pivoting a high-cardinality column creates thousands of columns.
- Missing combinations become null, not zero.

### Example

Pivot revenue per customer by status, then unpivot it back to one row per customer and status.

In [ ]:
STATUSES = ["placed", "paid", "shipped", "returned"]
wide = orders.groupBy("customer_id").pivot("status", STATUSES).agg(F.sum("amount"))
wide.orderBy("customer_id").show()

long = wide.unpivot("customer_id", STATUSES, "status", "revenue").where("revenue IS NOT NULL")
long.orderBy("customer_id", "status").show(5)

**What to notice:** Each customer is one row with a column per status; combinations that do not exist are null. `unpivot` produced
one row per customer and status again, and filtering out the nulls brings back exactly the combinations present
in the data.

## Subtotals with rollup, cube and grouping sets

- `rollup("region", "status")` adds subtotals per region and a grand total: hierarchical levels.
- `cube("region", "status")` adds every combination, including totals per status across regions.
- `GROUPING SETS` (SQL) lists exactly the levels you want.

In subtotal rows, the rolled-up column is null. `F.grouping_id()` (or `grouping(col)`) labels each row's level,
which is the only reliable way to tell a subtotal from a row whose key really is null.

**Where it matters**
- **Finance reports** with subtotals and a grand total from one query.
- **OLAP-style summaries** feeding BI tools.

**Watch out for**
- Never detect subtotals by testing for null: real null keys look the same.
- `cube` over n columns produces 2^n grouping levels.

### Example

Roll revenue up by region and status. Customer 4 has no region, so some detail rows have a null region too.

In [ ]:
with_region = orders.join(customers.select("customer_id", "region"), "customer_id", "left")
rolled = (
    with_region.rollup("region", "status")
    .agg(F.sum("amount").alias("revenue"), F.grouping_id().alias("level"))
    .orderBy("level", F.col("region").asc_nulls_last(), "status")
)
rolled.show(30)

spark.sql("""
    SELECT region, status, sum(amount) AS revenue, grouping_id() AS level
    FROM {df} GROUP BY GROUPING SETS ((region), (status), ())
    ORDER BY level, region, status
""", df=with_region).show()

**What to notice:** `level` 0 rows are detail, 1 are region subtotals, and 3 is the grand total (2,659.05). Customer 4's two orders
appear at level 0 with a null region, and again inside the null-region subtotal at level 1: without `level`,
those rows would be indistinguishable from subtotals. The grouping-sets query returns only the levels it lists:
per region, per status and the total.

## Window functions: compute over related rows

A **window function** computes a value for each row from a set of related rows, without collapsing them. The
window is defined with `Window.partitionBy(...)` (which rows belong together), `.orderBy(...)` (their order) and
optionally a frame (which neighbours each row sees). An aggregate over a window with no order, such as
`F.sum("amount").over(Window.partitionBy("customer_id"))`, puts the group total on every row.

Spark executes a window by shuffling rows by the partition key, sorting each partition, and evaluating all
window expressions that share the same specification in one pass.

**Where it matters**
- **Shares and ratios:** each order's share of its customer's revenue.
- **Per-row context:** compare a row with its group's average or maximum.

**Watch out for**
- A window with no `partitionBy` moves every row into one partition and one task.
- Each distinct window specification adds its own shuffle and sort; reuse one specification for several expressions.

### Example

Put each customer's total revenue and order count on every order, and compute the order's share.

In [ ]:
by_customer = Window.partitionBy("customer_id")
orders.select(
    "customer_id", "order_id", "amount",
    F.sum("amount").over(by_customer).alias("customer_total"),
    F.count("*").over(by_customer).alias("customer_orders"),
    F.round(F.col("amount") / F.sum("amount").over(by_customer), 2).alias("share"),
).orderBy("customer_id", "order_id").show()

**What to notice:** Every order is still there, unlike with `groupBy`, and each carries its customer's total. Customer 1's three
orders share 234.50 as 51, 36 and 13 percent. Customers with a single order show a share of 1.00.

## Rank rows and keep the top N per group

Ranking functions number rows within each partition by the window's order:

| function | values for 300, 300, 200 |
| --- | --- |
| `row_number` | 1, 2, 3 (ties in arbitrary order) |
| `rank` | 1, 1, 3 (gap after ties) |
| `dense_rank` | 1, 1, 2 (no gap) |

`ntile(n)` splits a partition into `n` buckets and `percent_rank` gives a position from 0 to 1. Filtering on a
rank gives the top N per group.

**Where it matters**
- **Top products per category** and best customers per region.
- **Exactly one row per key** with `row_number`.

**Watch out for**
- `row_number` over a non-unique order picks ties arbitrarily; add a tie-breaker column.
- "Top 2" with ties: decide whether you want two rows (`row_number`) or two values (`dense_rank`).

### Example

Two sales reps in the north tie at 300. Compare the three ranking functions on the same window.

In [ ]:
sales = spark.createDataFrame(
    [("north", "Ana", 300), ("north", "Ben", 300), ("north", "Cai", 200), ("north", "Dee", 100), ("south", "Eli", 500)],
    "region string, rep string, revenue int",
)
by_revenue = Window.partitionBy("region").orderBy(F.desc("revenue"))
sales.select(
    "region", "rep", "revenue",
    F.row_number().over(by_revenue).alias("row_number"),
    F.rank().over(by_revenue).alias("rank"),
    F.dense_rank().over(by_revenue).alias("dense_rank"),
    F.ntile(2).over(by_revenue).alias("half"),
).orderBy("region", "row_number").show()

**What to notice:** Ana and Ben tie at 300. `row_number` still numbers them 1 and 2, in an order you should not rely on. `rank` gives
both 1 and jumps to 3 for Cai; `dense_rank` gives both 1 and Cai 2. `ntile(2)` split the four north reps into two
halves.

### Example: top N per group

Keep the two best-selling products in each category.

In [ ]:
product_revenue = (
    order_items.join(products, "sku")
    .groupBy("category", "sku")
    .agg(F.sum(F.col("qty") * F.col("unit_price")).alias("revenue"))
)
best_first = Window.partitionBy("category").orderBy(F.desc("revenue"), "sku")
(
    product_revenue.withColumn("rank", F.dense_rank().over(best_first))
    .where("rank <= 2")
    .orderBy("category", "rank")
    .show()
)

**What to notice:** Two products per category, ranked within the category rather than across all products: Pantry's top product is
the 1,180.00 bulk SKU, while Kitchen's best sells 95.00. The window ranks inside each partition, so every category
gets its own top two in a single pass.

## Compare neighbouring rows with lag and lead

`F.lag(col, n)` reads the value `n` rows earlier in the window order; `F.lead(col, n)` reads `n` rows later. Both
return null (or a default you pass) at the edges of a partition. Combined with date functions they give gaps
between events and changes from the previous value.

**Where it matters**
- **Days between orders** for churn and repeat-purchase metrics.
- **Status transitions:** what was the previous status of this order?

**Watch out for**
- Without `partitionBy`, `lag` reads across customers and mixes their data.
- The first row of each partition has no previous row; decide what null should mean.

### Example

For each order, find the customer's previous order time, the days since it, and the next order id.

In [ ]:
in_order = Window.partitionBy("customer_id").orderBy("order_ts", "order_id")
orders.select(
    "customer_id", "order_id", "order_ts",
    F.lag("order_ts").over(in_order).alias("previous_ts"),
    F.datediff(F.to_date("order_ts"), F.to_date(F.lag("order_ts").over(in_order))).alias("days_since_previous"),
    F.lead("order_id").over(in_order).alias("next_order_id"),
).orderBy("customer_id", "order_ts").show()

**What to notice:** Each customer's first order has no previous time and the last has no next order. Customer 2 reordered after one
day and customer 3 after two; customer 1's three orders were all on the same day, so the gap is 0.

## Running totals and moving windows

A **frame** limits which rows of the partition each row sees:

- `rowsBetween(Window.unboundedPreceding, Window.currentRow)`: everything so far, a running total;
- `rowsBetween(-2, 0)`: this row and the two before it, a 3-row moving window;
- `rangeBetween(-7 * 86400, 0)` over a numeric ordering (seconds): every row within the last 7 days, however many.

When a window has `orderBy` but no frame, Spark uses `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`. That
is right for running totals but makes `last` return the current row.

**Where it matters**
- **Cumulative revenue** per customer or per day.
- **Trailing 7-day metrics** for fraud rules and dashboards.

**Watch out for**
- A rows frame over irregular timestamps mixes a one-day gap with a one-month gap; use a range frame for calendar windows.
- With the default range frame, rows with equal ordering values are all included together.

### Example

Compute a running total, a 3-row moving average and a trailing 7-day sum per customer; then see the default-frame trap with `last`.

In [ ]:
running = in_order.rowsBetween(Window.unboundedPreceding, Window.currentRow)
three_rows = in_order.rowsBetween(-2, 0)
seven_days = Window.partitionBy("customer_id").orderBy(F.col("order_ts").cast("long")).rangeBetween(-7 * 86400, 0)

orders.select(
    "customer_id", "order_id", "order_ts", "amount",
    F.sum("amount").over(running).alias("running_total"),
    F.round(F.avg("amount").over(three_rows), 2).alias("avg_last_3"),
    F.sum("amount").over(seven_days).alias("spend_7d"),
).where("customer_id IN (1, 2)").orderBy("customer_id", "order_ts").show()

whole = in_order.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
orders.where("customer_id = 1").select(
    "order_id",
    F.last("order_id").over(in_order).alias("last_default_frame"),
    F.last("order_id").over(whole).alias("last_whole_partition"),
).orderBy("order_id").show()

**What to notice:** Customer 1's running total grows 120.00, 204.50, 234.50, and order 5's 7-day spend includes order 4 from the day
before. In the second table, `last` with the default frame returns each row's own id, because the frame ends at
the current row; only the explicit whole-partition frame returns 3, the real last order.

## Keep the latest record per key

Change feeds and retries deliver several versions of the same key. Keep one row per key with `row_number` over a
window ordered by the **source** sequence (a log sequence number, a version, the source's `updated_at`), newest
first, plus a tie-breaker that is stable across reruns, and filter to `rn = 1`.

`dropDuplicates(["key"])` also keeps one row per key, but which row it keeps is not defined.

**Where it matters**
- **Silver tables** built from CDC feeds: one current row per order.
- **Replays:** a reprocessed batch must pick the same winners.

**Watch out for**
- Timestamps with second precision tie; without a tie-breaker the winner can change after a repartition.
- Ingestion time changes on replay; use it only as the last tie-breaker.

### Example

Order 101 has three versions; order 102's final change arrived twice. Keep the latest version of each.

In [ ]:
changes = spark.createDataFrame(
    [(101, "placed", 1, datetime(2026, 1, 5, 9, 0)), (101, "paid", 2, datetime(2026, 1, 5, 9, 5)),
     (101, "shipped", 3, datetime(2026, 1, 6, 8, 0)), (102, "placed", 1, datetime(2026, 1, 5, 10, 0)),
     (102, "cancelled", 2, datetime(2026, 1, 5, 10, 1)), (102, "cancelled", 2, datetime(2026, 1, 5, 10, 9))],
    "order_id int, status string, source_lsn long, ingested_at timestamp",
)
newest_first = Window.partitionBy("order_id").orderBy(F.desc("source_lsn"), F.desc("ingested_at"))
latest = changes.withColumn("rn", F.row_number().over(newest_first)).where("rn = 1").drop("rn")
latest.orderBy("order_id").show()

**What to notice:** Order 101 keeps its highest sequence, `shipped`. Order 102's two `cancelled` rows share sequence 2, so the
ingestion time breaks the tie and the 10:09 copy wins, on every run.

## Sessionize events

A **session** is a burst of activity separated by inactivity, for example 30 minutes without an event. The
classic recipe: use `lag` to compute the gap since the user's previous event, flag a new session when the gap is
larger than the timeout (or there is no previous event), then take a running sum of the flags to number the
sessions. Structured Streaming offers `session_window` for the same idea on streams ([lesson 11](pyspark-11-structured-streaming.ipynb)).

**Where it matters**
- **Web and app analytics:** sessions per user, session length, pages per session.
- **IoT:** periods of machine activity.

**Watch out for**
- Partition by user, or sessions leak from one user into the next.
- Deduplicate retried events first, or a retry can look like activity.

### Example

Split each user's events into sessions with a 30-minute timeout.

In [ ]:
user_time = Window.partitionBy("user_id").orderBy("ts")
sessions = (
    events.dropDuplicates()
    .withColumn("gap_minutes", (F.col("ts").cast("long") - F.lag("ts").over(user_time).cast("long")) / 60)
    .withColumn("new_session", F.when(F.col("gap_minutes").isNull() | (F.col("gap_minutes") > 30), 1).otherwise(0))
    .withColumn("session_number", F.sum("new_session").over(user_time.rowsBetween(Window.unboundedPreceding, 0)))
)
sessions.where("user_id IN (1, 3, 5)").orderBy("user_id", "ts").show()

sessions.groupBy("user_id", "session_number").agg(
    F.count("*").alias("events"), F.min("ts").alias("started"), F.max("ts").alias("ended")
).orderBy("user_id", "session_number").show()

**What to notice:** User 1's three events fall within 10 minutes, so they form one session. User 3's two views are two days apart:
two sessions. User 5's first view and the next morning's events are separate sessions, and the morning view and
purchase three and a half hours apart are separate again. The duplicate event was removed first, so it did not
create a zero-minute gap.

## Find consecutive runs: gaps and islands

"Longest streak of consecutive active days" is a **gaps-and-islands** problem. Number each user's distinct days
with `row_number`, and subtract that number from the date: every day in an unbroken run gives the same result,
so grouping by it finds each run (an island).

**Where it matters**
- **Loyalty and engagement:** login streaks, consecutive months with a purchase.
- **Operations:** periods of continuous downtime or uptime.

**Watch out for**
- Deduplicate to one row per user and day first, or a repeated day breaks the arithmetic.
- Decide how to break ties between equally long runs (the earliest, usually).

### Example

Find each user's longest run of consecutive days with at least one login.

In [ ]:
logins = spark.createDataFrame(
    [("u1", "2026-01-01"), ("u1", "2026-01-02"), ("u1", "2026-01-02"), ("u1", "2026-01-03"),
     ("u1", "2026-01-05"), ("u1", "2026-01-06"), ("u2", "2026-01-10"), ("u3", "2026-01-01"), ("u3", "2026-01-03")],
    "user_id string, login_date string",
)
days = logins.select("user_id", F.to_date("login_date").alias("day")).distinct()
islands = days.withColumn("island", F.date_sub("day", F.row_number().over(Window.partitionBy("user_id").orderBy("day"))))
islands.orderBy("user_id", "day").show()

streaks = islands.groupBy("user_id", "island").agg(
    F.count("*").alias("days"), F.min("day").alias("start"), F.max("day").alias("end")
)
longest = Window.partitionBy("user_id").orderBy(F.desc("days"), "start")
streaks.withColumn("rn", F.row_number().over(longest)).where("rn = 1").drop("rn", "island").orderBy("user_id").show()

**What to notice:** For user 1, 1 to 3 January all map to the same `island` date (31 December), and 5 and 6 January map to another,
so the longest streak is 3 days. The duplicate login on 2 January was removed by `distinct` before numbering.
User 3's two single days are two islands of 1; the earlier one wins the tie.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** Why compute several metrics in one `agg` rather than one query each?

**A.** One `agg` scans and shuffles the data once; separate queries scan and shuffle it once per metric.

**Q2.** A sum of returned revenue is null for some customers. Why?

**A.** The sum of no values is null. Wrap the aggregate in `coalesce(..., 0)` if the report should show zero.

**Q3.** Why pass explicit values to `pivot`?

**A.** Without them Spark runs an extra job to find the values, and a new value in the data silently adds a column.

**Q4.** How do you tell a rollup subtotal from a row whose key is null?

**A.** With `grouping_id()` or `grouping(col)`; the key is null in both.

**Q5.** What does `last(col)` return with `orderBy` but no frame, and why?

**A.** The current row's value: the default frame runs from the start of the partition to the current row.

**Q6.** Describe the gaps-and-islands trick.

**A.** Number each key's distinct days with `row_number` and subtract it from the date. Consecutive days produce the same value, so grouping by it finds each run.

**Q7.** How do you sessionize events with a 30-minute timeout?

**A.** Compute the gap to the previous event with `lag`, flag a new session when it exceeds 30 minutes or is null, and number sessions with a running sum of the flags, all partitioned by user.

## Key takeaways

- Aggregate many metrics in one `agg`; conditions go inside the aggregate.
- Approximate aggregates are for large-scale monitoring, not for money.
- Pass explicit values to `pivot`; label rollup levels with `grouping_id()`.
- Window functions keep every row: partition, order and frame decide what each row sees.
- Latest record per key: `row_number` over the source sequence plus a tie-breaker.
- Sessions and streaks are `lag`, running sums and `row_number` arithmetic, always partitioned by the entity.

**Next:** Practise this module in [PySpark drills 7: aggregations and windows](pyspark-drills-07-aggregations-and-windows.ipynb), then continue with [PySpark 8: nested and semi-structured data](pyspark-08-nested-and-semi-structured-data.ipynb).